# 06. 강건성 — 공격 주입·탐지·방어 (배혜림 D1~D7)

제안서 6절. 판매자가 가짜 계정으로 자기 상품 평점을 올리면 추천이 얼마나 망가지고,
탐지로 얼마나 되돌릴 수 있는지 본다.

| 셀 | 업무 | 내용 |
|---|---|---|
| 1 | — | 데이터, 05에서 고른 설정 읽기, 공격 없는 기준선 |
| 2 | — | **스모크 테스트 — 시나리오 1개로 시간·메모리 측정** |
| 3 | **D1~D7** | 전체 시나리오 |
| 4 | **D4~D6** | 탐지기 비교, 피처 중요도 |
| 5 | — | 회복 요약, 성공 기준 ⑤⑥ |
| 6 | **D7** | 편향 수축 방어 |
| 7 | — | 저장 |


### 설계 원칙

- **탐지기는 공격 전 데이터를 못 본다.** 상품 평균·평점 수, DegSim 참조 사용자를 모두
  오염된 데이터에서 다시 계산한다. 공격 전 깨끗한 스냅샷을 쓰면 현실에 없는 정보를
  쓰는 셈이라 F1이 부풀려진다. (`USE_CLEAN_STATS=True`로 바꾸면 비교 실험을 할 수 있다)
- **탐지 평가 표본은 전체 사용자에서 비례로 뽑는다.** 가짜만 전부 넣고 정상은 일부만
  뽑으면 양성 비율이 실제 공격 규모가 아니게 되어 F1을 성공 기준 ⑤에 쓸 수 없다.
- **제거는 전체 사용자에 적용한다.** 정상 사용자를 잘못 지우는 오탐까지 재학습에 반영한다.
- **가짜 프로필 길이는 실제 사용자 분포에서 뽑는다.** 길이를 고정하면 탐지기가 공격 행동이
  아니라 길이만 보고 맞힌다.
- **공격 대상 모델은 05에서 고른 MF와 같다.** `results/mf/{DS}_mf_config.json`을 읽어 쓴다.
  B3에서 잠재요인 기여가 0으로 확인되면 `ATTACK_K=0`으로 바꿔 시간을 줄여도 된다.
- **시드는 시나리오 번호로 만든다.** `hash()`는 실행할 때마다 달라져 재현이 안 된다.

In [4]:
# ===== 0. 설정 =====
import os, sys, json, subprocess, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/team10-recsys'
    if os.path.exists(REPO):
        subprocess.run(['git', '-C', REPO, 'pull', '-q'])
    else:
        subprocess.run(['git', 'clone', '-q',
                        'https://github.com/qhfl3644/team10-recsys.git', REPO])
    sys.path.insert(0, REPO)
except ImportError:
    REPO = os.path.abspath('..')
    sys.path.insert(0, REPO)

import numpy as np, pandas as pd
from config import RESULT_DIR, SEED
from src.eval import metrics as M
from src.models.mf import MF
from src.attack import attacks as A, detection as D, impact as I

DS = 'Video_Games'
EPOCHS = 10             # 공격 실험용 본 학습과 달리 짧게 설정
N_REF = 1500            # DegSim 참조 사용자 수 (전체 쌍 계산은 불가능)
N_HR = 2000             # 회복 측정 사용자 수 (타깃 미평가자 중에서)
N_EVAL = 15000          # 탐지 평가 표본 (전체에서 비례 추출, 양성 비율 보존)
USE_CLEAN_STATS = False # True면 탐지기가 공격 전 통계를 본다 (비교 실험용)
ATTACK_K = None         # None이면 05에서 고른 k를 읽어 쓴다. 0이면 편향만
print('준비 완료:', DS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
준비 완료: Video_Games


## 1. 데이터 + 05에서 고른 설정 + 기준선

In [5]:

# ===== 1. 기준선 =====
t0 = time.time()
data = M.load_data(DS)

# 05 노트북이 저장한 설정을 읽어 같은 모델로 공격 실험을 한다
cfg_path = f'{RESULT_DIR}/mf/{DS}_mf_config.json'
if not os.path.exists(cfg_path):
    raise FileNotFoundError(f'05 설정 파일이 없습니다: {cfg_path}')
with open(cfg_path, encoding='utf-8') as f:
    cfg = json.load(f)
MF_K, MF_LR, MF_REG = int(cfg['k']), float(cfg['lr']), float(cfg['reg'])
print(f'05 설정 사용: k={MF_K}, lr={MF_LR}, reg={MF_REG}')
if ATTACK_K is not None:
    MF_K = int(ATTACK_K); print(f'  → ATTACK_K 지정으로 k={MF_K} 사용')

def make_mf():
    return MF(k=MF_K, lr=MF_LR, reg=MF_REG, epochs=EPOCHS, patience=3, seed=SEED, verbose=False)

clean_stats = A.item_stats(data.train, data.n_items)      # 공격자는 공개 평점을 본다
mean_c, std_c, count_c, mu_c, sd_c = clean_stats
length_pool = A.user_length_pool(data.train)
targets = A.pick_targets(count_c)

rng0 = np.random.default_rng([SEED, 101])
hr_users = {lab: I.eligible_users(data, t, N_HR, rng0) for lab, t in targets.items()}

clean = make_mf().fit(data.train, data.n_users, data.n_items, valid=data.valid)
y = data.test['rating'].values
print(f'공격 없는 MF test RMSE '
      f'{M.rmse(y, clean.predict(data.test.uidx.values, data.test.iidx.values)):.4f}')

base_metrics = {}
for lab, t in targets.items():
    base_metrics[lab] = I.target_metrics(clean, t, hr_users[lab], data)
    print(f'  [{lab}] iidx={t}, train 평점 {count_c[t]}개, 미평가 사용자 {len(hr_users[lab]):,}명'
          f' → 점수 {base_metrics[lab][0]:.3f}, HR@10 {base_metrics[lab][1]:.3f}')
print(f'가짜 프로필 길이는 실제 분포에서 추출 (실제 중앙값 {np.median(length_pool):.0f}개)')
print(f'({time.time()-t0:.0f}초)')

05 설정 사용: k=16, lr=0.01, reg=0.1
공격 없는 MF test RMSE 1.1821
  [popular] iidx=15574, train 평점 2062개, 미평가 사용자 2,000명 → 점수 4.683, HR@10 0.000
  [unpopular] iidx=25611, train 평점 5개, 미평가 사용자 2,000명 → 점수 4.256, HR@10 0.000
가짜 프로필 길이는 실제 분포에서 추출 (실제 중앙값 4개)
(138초)


## 2. 한 시나리오 실행 함수 + 스모크 테스트

In [6]:
# ===== 2. 실행 함수 =====
def run_scenario(sid, kind, size, tlab, verbose=True):
    target = targets[tlab]
    n_fake = int(data.n_users * size)
    rs = np.random.default_rng([SEED, 201, sid])
    t = time.time(); stamp = {}

    # D1~D2 주입 (공격자는 공격 전 공개 통계를 본다 → clean_stats)
    tr2, fake = A.inject(data.train, data.n_users, data.n_items, target, kind,
                         n_fake, clean_stats, rs, length_pool=length_pool)
    nu2 = data.n_users + n_fake
    atk = make_mf().fit(tr2, nu2, data.n_items, valid=data.valid)
    stamp['주입+학습'] = time.time() - t; t = time.time()

    # D3 피처 — 방어자는 오염된 데이터만 본다
    if USE_CLEAN_STATS:
        d_mean, d_count = mean_c, count_c
    else:
        s2 = A.item_stats(tr2, data.n_items)
        d_mean, d_count = s2[0], s2[2]
    users = np.arange(nu2, dtype=np.int64)
    lab = (users >= data.n_users).astype(int)
    ref = np.sort(rs.choice(nu2, size=min(N_REF, nu2), replace=False))   # 가짜 포함
    F, names = D.features(tr2, users, data.n_items, d_mean, d_count, ref)
    stamp['피처'] = time.time() - t; t = time.time()

    # D4~D6 탐지 — 전체에서 비례 추출한 표본으로 평가 (양성 비율 보존)
    ev = np.sort(rs.choice(nu2, size=min(N_EVAL, nu2), replace=False))
    tab, imp, dt_oof = D.detect(F[ev], lab[ev], names)
    det = tab.set_index('탐지기').loc['Decision Tree']
    stamp['탐지'] = time.time() - t; t = time.time()

    # D7 제거 — 평가 표본에는 held-out 예측, 나머지에는 평가 표본으로 학습한 나무 적용
    from sklearn.tree import DecisionTreeClassifier
    flag_pred = np.zeros(nu2, dtype=int)
    flag_pred[ev] = dt_oof
    rest = np.setdiff1d(users, ev, assume_unique=True)
    if len(rest):
        final_dt = DecisionTreeClassifier(max_depth=6, class_weight='balanced',
                                          random_state=SEED).fit(F[ev], lab[ev])
        flag_pred[rest] = final_dt.predict(F[rest])
    flagged = users[flag_pred == 1]
    n_removed_real = int((flagged < data.n_users).sum())
    kept = tr2[~tr2['uidx'].isin(set(flagged.tolist()))]
    rec = make_mf().fit(kept, nu2, data.n_items, valid=data.valid)
    a = I.target_metrics(atk, target, hr_users[tlab], data)
    r = I.target_metrics(rec, target, hr_users[tlab], data)
    stamp['제거+재학습'] = time.time() - t

    row = I.recovery_row(kind, size, tlab, base_metrics[tlab], a, r, det, n_fake, n_removed_real)
    if verbose:
        print(f"  {kind:10s} {size:.0%} {tlab:9s} 양성 {lab[ev].mean()*100:.1f}%"
              f"  점수 {base_metrics[tlab][0]:.2f}→{a[0]:.2f}→{r[0]:.2f}"
              f"  F1 {det['F1']}  오탐 {n_removed_real:,}명")
    return row, (f'{kind} {size:.0%} {tlab}', tab, imp, float(lab[ev].mean())), stamp

In [7]:
# ===== 2-1. 스모크 테스트 (시나리오 1개) =====
t0 = time.time()
row, dt_, stamp = run_scenario(0, 'average', 0.03, 'unpopular')
el = time.time() - t0
print('\n단계별 소요(초):', {k: round(v, 1) for k, v in stamp.items()})
print(f'시나리오 1개 {el:.0f}초 → 전체 18개 단순 환산 약 {el*18/60:.0f}분 + 방어 실험')
try:
    import psutil
    print(f'메모리 사용 {psutil.Process().memory_info().rss/1e9:.1f} GB')
except ImportError:
    pass

  average    3% unpopular 양성 3.0%  점수 4.26→4.99→4.98  F1 0.805  오탐 1,342명

단계별 소요(초): {'주입+학습': 135.0, '피처': 8.0, '탐지': 1.7, '제거+재학습': 125.5}
시나리오 1개 270초 → 전체 18개 단순 환산 약 81분 + 방어 실험
메모리 사용 0.6 GB


## 3. 전체 시나리오 (D1~D7)

In [9]:
# ===== 3. 전체 시나리오 =====
# 공격 3종 × 규모 3단계 × 타깃 2종 = 18개 전체 조합
SCEN = [(kind, size, tlab)
        for kind in A.KINDS
        for size in (0.01, 0.03, 0.05)
        for tlab in ('unpopular', 'popular')]
print('전체 시나리오:', len(SCEN), '개')

t0 = time.time(); rows, det_tables = [], []
for sid, (kind, size, tlab) in enumerate(SCEN):
    row, dtab, _ = run_scenario(sid, kind, size, tlab)
    rows.append(row); det_tables.append(dtab)
    print(f'    누적 {time.time()-t0:.0f}초')
recovery = pd.DataFrame(rows)

전체 시나리오: 18 개
  random     1% unpopular 양성 1.0%  점수 4.26→4.98→5.00  F1 0.295  오탐 3,213명
    누적 265초
  random     1% popular   양성 1.0%  점수 4.68→4.79→4.70  F1 0.443  오탐 1,840명
    누적 530초
  random     3% unpopular 양성 2.8%  점수 4.26→4.99→4.98  F1 0.714  오탐 1,763명
    누적 803초
  random     3% popular   양성 3.0%  점수 4.68→4.79→4.48  F1 0.748  오탐 1,958명
    누적 1075초
  random     5% unpopular 양성 5.0%  점수 4.26→4.98→4.99  F1 0.888  오탐 872명
    누적 1353초
  random     5% popular   양성 4.8%  점수 4.68→4.93→4.35  F1 0.824  오탐 1,646명
    누적 1629초
  average    1% unpopular 양성 1.0%  점수 4.26→4.98→4.97  F1 0.289  오탐 3,996명
    누적 1892초
  average    1% popular   양성 0.9%  점수 4.68→4.70→4.55  F1 0.398  오탐 2,611명
    누적 2155초
  average    3% unpopular 양성 2.9%  점수 4.26→4.97→4.95  F1 0.829  오탐 683명
    누적 2427초
  average    3% popular   양성 3.1%  점수 4.68→4.89→4.49  F1 0.721  오탐 2,254명
    누적 2697초
  average    5% unpopular 양성 4.6%  점수 4.26→4.99→4.96  F1 0.899  오탐 1,164명
    누적 2975초
  average    5% popular   양성 4.8%  점

## 4. 탐지기 비교 + 피처 중요도 (D4~D6)

In [10]:
# ===== 4. 탐지기 =====
# 공격 방식 비교가 가능한 3%·비인기 타깃 세 시나리오를 화면에 표시
representative = [x for x in det_tables if '3% unpopular' in x[0]]
for label, tab, imp, rate in representative:
    print(f'[{label}]  양성 비율 {rate*100:.1f}%  무작위 F1 상한 {D.random_f1_bound(rate):.3f}')
    print(tab.to_string(index=False))
    print('  중요 피처:', ', '.join(f'{k}({v:.2f})' for k, v in imp[:4]), '\n')

[random 3% unpopular]  양성 비율 2.8%  무작위 F1 상한 0.055
          탐지기   정밀도   재현율    F1  TP   FP  FN    TN
Decision Tree 0.576 0.938 0.714 395  291  26 14288
          kNN 0.837 0.831 0.834 350   68  71 14511
      k-means 0.032 0.283 0.058 119 3593 302 10986
  중요 피처: DegSim(0.76), WDA(0.10), WDMA(0.09), 평균 절대 편차(0.03) 

[average 3% unpopular]  양성 비율 2.9%  무작위 F1 상한 0.057
          탐지기   정밀도   재현율    F1  TP    FP  FN    TN
Decision Tree 0.746 0.932 0.829 411   140  30 14419
          kNN 0.947 0.932 0.939 411    23  30 14536
      k-means 0.032 0.800 0.061 353 10725  88  3834
  중요 피처: DegSim(0.78), WDMA(0.07), 프로필 길이(0.05), WDA(0.05) 

[bandwagon 3% unpopular]  양성 비율 2.9%  무작위 F1 상한 0.057
          탐지기   정밀도   재현율    F1  TP    FP  FN    TN
Decision Tree 0.799 0.959 0.872 421   106  18 14455
          kNN 0.949 0.938 0.944 412    22  27 14539
      k-means 0.038 0.952 0.074 418 10497  21  4064
  중요 피처: DegSim(0.78), 프로필 길이(0.10), 평균 평점(0.05), WDA(0.04) 



## 5. 회복 요약 + 성공 기준 ⑤⑥

- 성공 기준 ⑤: 공격 규모 3% 이상에서 의사결정나무 F1 ≥ 0.7
- 성공 기준 ⑥: 회복 정도는 판정하지 않고 타깃 인기도별로 보고

In [11]:
# ===== 5. 요약 =====
print(recovery.to_string(index=False))

big = recovery[recovery['규모'].str.rstrip('%').astype(int) >= 3]
ok = bool((big['DT_F1'] >= 0.7).all())
rates3 = [r for label, _, _, r in det_tables if ' 3% ' in label]
rate3 = float(np.mean(rates3))
print(f"\n[성공 기준 ⑤] {'통과' if ok else '실패'}  "
      f"(3% 이상 시나리오 F1 최저 {big['DT_F1'].min():.3f} / 기준 0.7)")
print(f"  평가 표본 양성 비율 {rate3*100:.1f}% → 무작위 분류기 F1 상한 {D.random_f1_bound(rate3):.3f}")
print("\n[성공 기준 ⑥] 타깃 인기도·탐지 재현율별 회복")
print('회복률: 1=완전 회복, 0=회복 없음, 음수=방어 후 더 악화')
print(recovery.groupby('타깃')[['DT_재현율', '상승폭', '잔존폭', '점수_회복률',
                                 'HR_공격후', 'HR_제거후', 'HR_회복률', '오탐_정상제거']]
      .mean().round(3).to_string())

       공격 규모  가짜수        타깃  점수_정상  점수_공격후  점수_제거후    상승폭    잔존폭  점수_회복률  HR_정상  HR_공격후  HR_제거후  HR_회복률  DT_F1  DT_정밀도  DT_재현율  오탐_정상제거
   random 1%  947 unpopular  4.256   4.978   4.996  0.722  0.740  -0.026    0.0   1.000     1.0     0.0  0.295   0.180   0.819     3213
   random 1%  947   popular  4.683   4.795   4.695  0.112  0.012   0.889    0.0   0.000     0.0     NaN  0.443   0.300   0.847     1840
   random 3% 2842 unpopular  4.256   4.986   4.976  0.730  0.720   0.015    0.0   1.000     1.0     0.0  0.714   0.576   0.938     1763
   random 3% 2842   popular  4.683   4.790   4.482  0.107 -0.201  -0.875    0.0   0.000     0.0     NaN  0.748   0.623   0.934     1958
   random 5% 4738 unpopular  4.256   4.981   4.994  0.725  0.738  -0.018    0.0   1.000     1.0     0.0  0.888   0.822   0.964      872
   random 5% 4738   popular  4.683   4.925   4.354  0.242 -0.328  -0.355    0.0   1.000     0.0     1.0  0.824   0.721   0.961     1646
  average 1%  947 unpopular  4.256   4.976   4.9

## 6. 편향 수축 방어 (D7)

탐지·제거 없이 **상품 편향의 정규화만 강하게** 걸어 공격 효과가 줄어드는지 확인

상품 편향을 `평점 합 / (정규화 상수 + 평점 개수)` 형태로 두면 평점 수가 적은 상품일수록 전역 평균 쪽으로 더 강하게 수축되어, 가짜 평점 몇 개가 예측을 덜 흔듦

In [12]:
# ===== 6. 편향 수축 방어 =====
t0 = time.time()
target = targets['unpopular']
n_fake = int(data.n_users * 0.03)
tr2, _ = A.inject(data.train, data.n_users, data.n_items, target, 'average',
                  n_fake, clean_stats, np.random.default_rng([SEED, 301]), length_pool=length_pool)
out = []
for lam in [MF_REG, 5.0, 25.0]:
    cm = MF(k=MF_K, lr=MF_LR, reg=MF_REG, reg_bi=lam, epochs=EPOCHS, patience=3,
            seed=SEED, verbose=False).fit(data.train, data.n_users, data.n_items, valid=data.valid)
    am = MF(k=MF_K, lr=MF_LR, reg=MF_REG, reg_bi=lam, epochs=EPOCHS, patience=3,
            seed=SEED, verbose=False).fit(tr2, data.n_users + n_fake, data.n_items, valid=data.valid)
    sc_, _ = I.target_metrics(cm, target, hr_users['unpopular'], data)
    sa_, ha_ = I.target_metrics(am, target, hr_users['unpopular'], data)
    out.append(dict(정규화상수=lam, 점수_정상=round(sc_, 3), 점수_공격후=round(sa_, 3),
                    상승폭=round(sa_ - sc_, 3), HR_공격후=round(ha_, 3),
                    test_RMSE=round(M.rmse(y, cm.predict(data.test.uidx.values, data.test.iidx.values)), 4)))
    print(f'  상수 {lam}: 상승폭 {out[-1]["상승폭"]:.3f}, RMSE {out[-1]["test_RMSE"]:.4f}  ({time.time()-t0:.0f}초)')

defense = pd.DataFrame(out)
print()
print(defense.to_string(index=False))
print('\n정규화 상수를 키웠을 때 상승폭(공격 효과)과 test_RMSE(평점 정확도)가 각각 어떻게 움직이는지 비교')

  상수 0.1: 상승폭 0.719, RMSE 1.1821  (263초)
  상수 5.0: 상승폭 0.135, RMSE 1.2055  (524초)
  상수 25.0: 상승폭 0.033, RMSE 1.2156  (788초)

 정규화상수  점수_정상  점수_공격후   상승폭  HR_공격후  test_RMSE
   0.1  4.256   4.975 0.719     1.0     1.1821
   5.0  4.273   4.408 0.135     0.0     1.2055
  25.0  4.280   4.313 0.033     0.0     1.2156

정규화 상수를 키웠을 때 상승폭(공격 효과)과 test_RMSE(평점 정확도)가 각각 어떻게 움직이는지 비교


## 7. 저장

In [13]:
# ===== 7. 저장 =====
OUT = f'{RESULT_DIR}/attack'; os.makedirs(OUT, exist_ok=True)
recovery_out = recovery.copy(); recovery_out.insert(0, 'dataset', DS)
defense_out = defense.copy(); defense_out.insert(0, 'dataset', DS)
recovery_out.to_csv(f'{OUT}/{DS}_recovery.csv', index=False)
defense_out.to_csv(f'{OUT}/{DS}_defense.csv', index=False)
pd.concat([t.assign(시나리오=l, 양성비율=r) for l, t, _, r in det_tables]).to_csv(
    f'{OUT}/{DS}_detectors.csv', index=False)
pd.DataFrame([dict(시나리오=l, 피처=k, 중요도=round(v, 4)) for l, _, im, _ in det_tables for k, v in im]
             ).to_csv(f'{OUT}/{DS}_feature_importance.csv', index=False)
M.save_json(dict(mf=dict(k=MF_K, lr=MF_LR, reg=MF_REG, epochs=EPOCHS),
                 n_ref=N_REF, n_hr=N_HR, n_eval=N_EVAL,
                 use_clean_stats=USE_CLEAN_STATS,
                 profile_length='실제 사용자 train 평점 수 분포에서 추출',
                 detection_population='전체 사용자에서 비례 추출',
                 scenario_design='공격 3종 × 규모 3단계 × 타깃 2종 전체 조합',
                 n_scenarios=len(SCEN),
                 criterion5_passed=ok), f'{OUT}/{DS}_attack_config.json')
print('저장 완료:', OUT)

저장 완료: /content/drive/MyDrive/Team10_RecSys/results/attack
